Two-Stage LSTM Satellite-Enhanced Rainfall Prediction Pipeline
Trains a Two-Stage forecasting framework (Stage 1: Rain Occurrence, Stage 2: Rain Amount)
by incorporating satellite variables from ERA5 Land, GSMaP, IMERG, and Oya datasets.
Configurable for local and Kaggle environments with LOCAL_TEST and FULL_TRAIN modes.


In [1]:
import os
import sys
import random
import warnings
import logging
import json
import glob
import logging
from pathlib import Path

# Data and ML libraries
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for server/script runs
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import optuna

# Scikit-Learn
from sklearn.preprocessing import MinMaxScaler
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    roc_auc_score, 
    brier_score_loss, 
    confusion_matrix, 
    log_loss, 
    mean_squared_error, 
    mean_absolute_error, 
    r2_score, 
    precision_recall_curve, 
    roc_curve, 
    auc
)
from sklearn.calibration import calibration_curve

# TensorFlow / Keras
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

2026-06-23 15:44:52.330222: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782229492.550519      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782229492.615752      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782229493.178897      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782229493.178950      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782229493.178954      16 computation_placer.cc:177] computation placer alr

# 1. RUN CONFIGURATION & ENVIRONMENT SETUP


In [2]:
RUN_MODE = "FULL_TRAIN"  # Options: "LOCAL_TEST" or "FULL_TRAIN"

# Seed configuration
SEED = 42
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

seed_everything(SEED)

# Logging configuration
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s', handlers=[logging.StreamHandler(sys.stdout)])
logger = logging.getLogger(__name__)
logger.info(f"Starting Satellite-Enhanced LSTM Rainfall Prediction Pipeline in {RUN_MODE} mode.")

# Dynamically set training parameters based on RUN_MODE
if RUN_MODE == "LOCAL_TEST":
    EPOCHS_OCC = 1
    EPOCHS_REG = 1
    OPTUNA_TRIALS = 1
    BATCH_SIZE = 64
else:
    EPOCHS_OCC = 50
    EPOCHS_REG = 200
    OPTUNA_TRIALS = 50
    BATCH_SIZE = 128

# Keras Embedding Layer Hot-Patch to prevent serialization issues
try:
    import keras as keras_standalone
    if hasattr(keras_standalone.layers, 'Embedding'):
        orig_init = keras_standalone.layers.Embedding.__init__
        def patched_init(self, *args, **kwargs):
            kwargs.pop('quantization_config', None)
            orig_init(self, *args, **kwargs)
        keras_standalone.layers.Embedding.__init__ = patched_init
except Exception:
    pass

try:
    if hasattr(tf.keras.layers, 'Embedding'):
        orig_init_tf = tf.keras.layers.Embedding.__init__
        def patched_init_tf(self, *args, **kwargs):
            kwargs.pop('quantization_config', None)
            orig_init_tf(self, *args, **kwargs)
        tf.keras.layers.Embedding.__init__ = patched_init_tf
except Exception:
    pass

# Dynamic path resolution
IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle')
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
BASE_DIR = os.path.abspath(os.path.join(SCRIPT_DIR, '..'))

if IS_KAGGLE:
    logger.info("Kaggle environment detected.")
    OUTPUTS_DIR = Path('/kaggle/working/outputs')
else:
    logger.info("Local environment detected.")
    OUTPUTS_DIR = Path(os.path.join(SCRIPT_DIR, 'outputs'))

# Create directories
os.makedirs(OUTPUTS_DIR / 'plots', exist_ok=True)
os.makedirs(OUTPUTS_DIR / 'metrics', exist_ok=True)
os.makedirs(OUTPUTS_DIR / 'model', exist_ok=True)

def find_file(filename, default_local_path):
    if IS_KAGGLE:
        matches = glob.glob(f'/kaggle/input/**/{filename}', recursive=True)
        if matches:
            return matches[0]
        # Direct fallback guess
        fallback_map = {
            'cuaca_jerukagung.csv': '/kaggle/input/datasets/jerismeteo/open-meteo-data-kebumen/open_meteo_jerukagung/cuaca_jerukagung.csv',
            'ERA5_Land_Standard_Units_TimeSeries_UTC_WMO.csv': '/kaggle/input/datasets/jerismeteo/google-earth-engine-data/ERA5_Land_Standard_Units_TimeSeries_UTC_WMO.csv',
            'Rainfall_GSMaP_TimeSeries_UNIX.csv': '/kaggle/input/datasets/jerismeteo/google-earth-engine-data/Rainfall_GSMaP_TimeSeries_UNIX.csv',
            'Rainfall_IMERG_TimeSeries_UNIX.csv': '/kaggle/input/datasets/jerismeteo/google-earth-engine-data/Rainfall_IMERG_TimeSeries_UNIX.csv',
            'Rainfall_Oya_TimeSeries_UNIX.csv': '/kaggle/input/datasets/jerismeteo/google-earth-engine-data/Rainfall_Oya_TimeSeries_UNIX.csv'
        }
        if filename in fallback_map and os.path.exists(fallback_map[filename]):
            return fallback_map[filename]
    return default_local_path

# Resolve paths
stational_path = find_file('cuaca_jerukagung.csv', os.path.join(BASE_DIR, 'Analisis_Meteorologi', 'open_meteo_jerukagung', 'cuaca_jerukagung.csv'))
era5_path = find_file('ERA5_Land_Standard_Units_TimeSeries_UTC_WMO.csv', os.path.join(BASE_DIR, 'Analisis_Meteorologi', 'Data_Satelit', 'ERA5_Land_Standard_Units_TimeSeries_UTC_WMO.csv'))
gsmap_path = find_file('Rainfall_GSMaP_TimeSeries_UNIX.csv', os.path.join(BASE_DIR, 'Analisis_Meteorologi', 'Data_Satelit', 'Rainfall_GSMaP_TimeSeries_UNIX.csv'))
imerg_path = find_file('Rainfall_IMERG_TimeSeries_UNIX.csv', os.path.join(BASE_DIR, 'Analisis_Meteorologi', 'Data_Satelit', 'Rainfall_IMERG_TimeSeries_UNIX.csv'))
oya_path = find_file('Rainfall_Oya_TimeSeries_UNIX.csv', os.path.join(BASE_DIR, 'Analisis_Meteorologi', 'Data_Satelit', 'Rainfall_Oya_TimeSeries_UNIX.csv'))

for name, p in [('Stational', stational_path), ('ERA5 Land', era5_path), ('GSMaP', gsmap_path), ('IMERG', imerg_path), ('Oya', oya_path)]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Required dataset not found for {name} at: {p}")
    logger.info(f"Resolved path for {name}: {p}")


2026-06-23 15:45:07,688 - INFO - Starting Satellite-Enhanced LSTM Rainfall Prediction Pipeline in FULL_TRAIN mode.
2026-06-23 15:45:07,689 - INFO - Kaggle environment detected.
2026-06-23 15:45:07,730 - INFO - Resolved path for Stational: /kaggle/input/datasets/jerismeteo/open-meteo-data-kebumen/open_meteo_jerukagung/cuaca_jerukagung.csv
2026-06-23 15:45:07,731 - INFO - Resolved path for ERA5 Land: /kaggle/input/datasets/jerismeteo/google-earth-engine-data/ERA5_Land_Standard_Units_TimeSeries_UTC_WMO.csv
2026-06-23 15:45:07,732 - INFO - Resolved path for GSMaP: /kaggle/input/datasets/jerismeteo/google-earth-engine-data/Rainfall_GSMaP_TimeSeries_UNIX.csv
2026-06-23 15:45:07,733 - INFO - Resolved path for IMERG: /kaggle/input/datasets/jerismeteo/google-earth-engine-data/Rainfall_IMERG_TimeSeries_UNIX.csv
2026-06-23 15:45:07,734 - INFO - Resolved path for Oya: /kaggle/input/datasets/jerismeteo/google-earth-engine-data/Rainfall_Oya_TimeSeries_UNIX.csv


# 2. DATA INGESTION, TIME ALIGNMENT, & MERGING


In [3]:
logger.info("Ingesting and aligning stational and satellite datasets...")

# Helper to read satellite data and localize UTC to WIB (Asia/Jakarta) timezone
def load_satellite_file(path):
    df = pd.read_csv(path)
    df['datetime'] = pd.to_datetime(df['datetime_utc'], utc=True).dt.tz_convert('Asia/Jakarta').dt.tz_localize(None)
    df = df.set_index('datetime').sort_index()
    return df

# A. ERA5 Land
logger.info("Processing ERA5 Land...")
df_era5 = load_satellite_file(era5_path).drop(columns=['unixtime', 'datetime_utc'], errors='ignore')
df_era5.columns = [f'sat_era5_{c}' for c in df_era5.columns]

# B. GSMaP
logger.info("Processing GSMaP...")
df_gsmap = load_satellite_file(gsmap_path).drop(columns=['unixtime', 'datetime_utc'], errors='ignore')
df_gsmap.columns = [f'sat_gsmap_{c}' for c in df_gsmap.columns]

# C. IMERG (30-min to hourly by sum)
logger.info("Processing IMERG...")
df_imerg_raw = load_satellite_file(imerg_path)
df_imerg = df_imerg_raw.drop(columns=['unixtime', 'datetime_utc'], errors='ignore').resample('1h').sum()
df_imerg.columns = [f'sat_imerg_{c}' for c in df_imerg.columns]

# D. Oya (30-min to hourly by sum)
logger.info("Processing Oya...")
df_oya_raw = load_satellite_file(oya_path)
df_oya = df_oya_raw.drop(columns=['unixtime', 'datetime_utc'], errors='ignore').resample('1h').sum()
df_oya.columns = [f'sat_oya_{c}' for c in df_oya.columns]

# Merge satellite data
logger.info("Merging satellite datasets...")
sat_combined = df_era5.join([df_gsmap, df_imerg, df_oya], how='outer')
sat_combined = sat_combined.interpolate(method='linear').ffill().bfill()

# E. Stational
logger.info("Processing Stational Data...")
df_stational = pd.read_csv(stational_path)
df_stational['datetime'] = pd.to_datetime(df_stational['datetime'], utc=True).dt.tz_convert('Asia/Jakarta').dt.tz_localize(None)
df_stational = df_stational.set_index('datetime').sort_index()

essential_cols = [
    'temperature_2m', 'relative_humidity_2m', 'dew_point_2m', 'rain', 
    'wind_speed_10m', 'wind_gusts_10m', 'wind_direction_10m', 'surface_pressure', 
    'sunshine_duration', 'shortwave_radiation', 'wet_bulb_temperature_2m', 'vapour_pressure_deficit'
]
cols_to_keep = [c for c in essential_cols if c in df_stational.columns]
df_stational = df_stational[cols_to_keep]
if 'rain' in df_stational.columns:
    df_stational.loc[df_stational['rain'] < 0, 'rain'] = 0

# F. Final Join (Keep only valid data after Oya's start date aligned at 2005-01-01)
logger.info("Aligning stational and satellite datasets...")
df_merged = df_stational.join(sat_combined, how='inner')
df_merged = df_merged.loc['2005-01-01 00:00:00':]
df_merged = df_merged.interpolate(method='linear').ffill().bfill()
logger.info(f"Aligned dataset size: {df_merged.shape[0]:,} rows.")


2026-06-23 15:45:07,766 - INFO - Ingesting and aligning stational and satellite datasets...
2026-06-23 15:45:07,768 - INFO - Processing ERA5 Land...
2026-06-23 15:45:08,639 - INFO - Processing GSMaP...
2026-06-23 15:45:09,038 - INFO - Processing IMERG...
2026-06-23 15:45:09,742 - INFO - Processing Oya...
2026-06-23 15:45:10,325 - INFO - Merging satellite datasets...
2026-06-23 15:45:10,442 - INFO - Processing Stational Data...
2026-06-23 15:45:12,906 - INFO - Aligning stational and satellite datasets...
2026-06-23 15:45:13,071 - INFO - Aligned dataset size: 187,687 rows.


# 3. FEATURE ENGINEERING & 3-HOURLY RESAMPLING


In [4]:
logger.info("Engineering features...")
df_feat = df_merged.copy()

# A. Stational U/V Wind Vectors
if 'wind_speed_10m' in df_feat.columns and 'wind_direction_10m' in df_feat.columns:
    wd_rad = df_feat['wind_direction_10m'] * np.pi / 180.0
    df_feat['wind_u'] = -df_feat['wind_speed_10m'] * np.sin(wd_rad)
    df_feat['wind_v'] = -df_feat['wind_speed_10m'] * np.cos(wd_rad)
    df_feat = df_feat.drop(columns=['wind_speed_10m', 'wind_direction_10m'])

# B. Stational Dewpoint Depression
if 'temperature_2m' in df_feat.columns and 'dew_point_2m' in df_feat.columns:
    df_feat['dewpoint_depression'] = df_feat['temperature_2m'] - df_feat['dew_point_2m']

# C. Lags for Stational Data
if 'rain' in df_feat.columns:
    for lag in [1, 2, 3, 6, 12, 24]:
        df_feat[f'rain_lag_{lag}'] = df_feat['rain'].shift(lag)

for lag in [1, 3, 6]:
    if 'relative_humidity_2m' in df_feat.columns:
        df_feat[f'humidity_lag_{lag}'] = df_feat['relative_humidity_2m'].shift(lag)
    if 'surface_pressure' in df_feat.columns:
        df_feat[f'pressure_lag_{lag}'] = df_feat['surface_pressure'].shift(lag)
    if 'temperature_2m' in df_feat.columns:
        df_feat[f'temperature_lag_{lag}'] = df_feat['temperature_2m'].shift(lag)

for lag in [1, 3]:
    if 'wind_u' in df_feat.columns:
        df_feat[f'wind_u_lag_{lag}'] = df_feat['wind_u'].shift(lag)
    if 'wind_v' in df_feat.columns:
        df_feat[f'wind_v_lag_{lag}'] = df_feat['wind_v'].shift(lag)
    if 'wind_gusts_10m' in df_feat.columns:
        df_feat[f'wind_gust_lag_{lag}'] = df_feat['wind_gusts_10m'].shift(lag)

# D. Trends/Diffs for Stational Data
for t in [1, 3]:
    if 'temperature_2m' in df_feat.columns:
        df_feat[f'temperature_change_{t}h'] = df_feat['temperature_2m'].diff(t)
    if 'relative_humidity_2m' in df_feat.columns:
        df_feat[f'humidity_change_{t}h'] = df_feat['relative_humidity_2m'].diff(t)
    if 'surface_pressure' in df_feat.columns:
        df_feat[f'pressure_change_{t}h'] = df_feat['surface_pressure'].diff(t)

if 'wind_u' in df_feat.columns:
    df_feat['wind_u_change_1h'] = df_feat['wind_u'].diff(1)
    df_feat['wind_v_change_1h'] = df_feat['wind_v'].diff(1)

# E. Rolling Features for Stational Data
windows = [3, 6, 12, 24]
roll_cols = [c for c in ['rain', 'relative_humidity_2m', 'surface_pressure', 'temperature_2m'] if c in df_feat.columns]
for col in roll_cols:
    for w in windows:
        df_feat[f'{col}_mean_{w}h'] = df_feat[col].rolling(w, min_periods=1).mean()
        df_feat[f'{col}_std_{w}h'] = df_feat[col].rolling(w, min_periods=1).std().fillna(0)
        df_feat[f'{col}_min_{w}h'] = df_feat[col].rolling(w, min_periods=1).min()
        df_feat[f'{col}_max_{w}h'] = df_feat[col].rolling(w, min_periods=1).max()

# F. Targeted Feature Engineering for Satellite Rain Features
sat_rain_cols = [
    'sat_gsmap_hourlyPrecipRate', 'sat_gsmap_hourlyPrecipRateGC', 
    'sat_imerg_precipitation_mmhr', 'sat_oya_precipitation_mmhr', 
    'sat_era5_total_precipitation_hourly_mm'
]
sat_rain_cols = [c for c in sat_rain_cols if c in df_feat.columns]

logger.info(f"Generating lags and rolling stats for satellite rain columns: {sat_rain_cols}")
for col in sat_rain_cols:
    # Lags (1, 2, 3, 6, 12, 24)
    for lag in [1, 2, 3, 6, 12, 24]:
        df_feat[f'{col}_lag_{lag}'] = df_feat[col].shift(lag)
    # Rolling mean, std, max over (3h, 6h, 12h, 24h)
    for w in [3, 6, 12, 24]:
        df_feat[f'{col}_roll_mean_{w}h'] = df_feat[col].rolling(window=w, min_periods=1).mean()
        df_feat[f'{col}_roll_std_{w}h'] = df_feat[col].rolling(window=w, min_periods=1).std().fillna(0)
        df_feat[f'{col}_roll_max_{w}h'] = df_feat[col].rolling(window=w, min_periods=1).max()

# G. Cyclical Time Features
df_feat['sin_hour'] = np.sin(2 * np.pi * df_feat.index.hour / 24.0)
df_feat['cos_hour'] = np.cos(2 * np.pi * df_feat.index.hour / 24.0)
df_feat['sin_doy'] = np.sin(2 * np.pi * df_feat.index.dayofyear / 365.25)
df_feat['cos_doy'] = np.cos(2 * np.pi * df_feat.index.dayofyear / 365.25)
df_feat['sin_month'] = np.sin(2 * np.pi * df_feat.index.month / 12.0)
df_feat['cos_month'] = np.cos(2 * np.pi * df_feat.index.month / 12.0)

df_feat = df_feat.dropna()

# H. 3-Hourly Resampling
logger.info("Resampling dataset to 3-hourly forecasting blocks...")
agg_rules = {}
for col in df_feat.columns:
    if 'rain' in col.lower() or 'precip' in col.lower():
        agg_rules[col] = 'sum'
    else:
        agg_rules[col] = 'mean'

df_3h = df_feat.resample('3h').agg(agg_rules).dropna()

# Create targets
df_3h['target_amount'] = df_3h['rain'].shift(-1)
df_3h = df_3h.dropna()
df_3h['target_occurrence'] = (df_3h['target_amount'] > 0).astype(int)

logger.info(f"Total resampled samples: {df_3h.shape[0]:,}. Features: {df_3h.shape[1]-2}")


2026-06-23 15:45:13,115 - INFO - Engineering features...
2026-06-23 15:45:13,719 - INFO - Generating lags and rolling stats for satellite rain columns: ['sat_gsmap_hourlyPrecipRate', 'sat_gsmap_hourlyPrecipRateGC', 'sat_imerg_precipitation_mmhr', 'sat_oya_precipitation_mmhr', 'sat_era5_total_precipitation_hourly_mm']


/tmp/ipykernel_16/3748512159.py:71: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_feat[f'{col}_lag_{lag}'] = df_feat[col].shift(lag)
/tmp/ipykernel_16/3748512159.py:71: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_feat[f'{col}_lag_{lag}'] = df_feat[col].shift(lag)
/tmp/ipykernel_16/3748512159.py:71: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a 

2026-06-23 15:45:14,768 - INFO - Resampling dataset to 3-hourly forecasting blocks...
2026-06-23 15:45:15,681 - INFO - Total resampled samples: 62,554. Features: 212


# 4. CHRONOLOGICAL SPLIT & SCALING


In [5]:
logger.info("Splitting dataset chronologically...")
train_mask = (df_3h.index.year >= 2005) & (df_3h.index.year <= 2023)
val_mask = (df_3h.index.year == 2024)
test_mask = (df_3h.index.year == 2025)

# Fallback split if years are empty
if df_3h[train_mask].empty:
    train_mask = df_3h.index < df_3h.index[int(len(df_3h) * 0.7)]
    val_mask = (df_3h.index >= df_3h.index[int(len(df_3h) * 0.7)]) & (df_3h.index < df_3h.index[int(len(df_3h) * 0.85)])
    test_mask = df_3h.index >= df_3h.index[int(len(df_3h) * 0.85)]

X = df_3h.drop(columns=['target_amount', 'target_occurrence'])
y = df_3h[['target_amount', 'target_occurrence']]
feature_names = X.columns.tolist()

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

# Sliced regression subsets (amount predicting trained ONLY on rainy instances target_occurrence == 1)
train_rain_mask = y_train['target_occurrence'] == 1
val_rain_mask = y_val['target_occurrence'] == 1
test_rain_mask = y_test['target_occurrence'] == 1

X_train_reg, y_train_reg = X_train[train_rain_mask], y_train[train_rain_mask]
X_val_reg, y_val_reg = X_val[val_rain_mask], y_val[val_rain_mask]
X_test_reg, y_test_reg = X_test[test_rain_mask], y_test[test_rain_mask]

logger.info(f"Train samples (Occ): {X_train.shape[0]:,}, Train samples (Reg, Rain > 0): {X_train_reg.shape[0]:,}")

# Fit scaler
scaler = MinMaxScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

X_train_reg_s = scaler.transform(X_train_reg)
X_val_reg_s = scaler.transform(X_val_reg)
X_test_reg_s = scaler.transform(X_test_reg)

# Save the fitted scaler
joblib.dump(scaler, OUTPUTS_DIR / 'model' / 'scaler_prediction.pkl')
logger.info("Scaler exported successfully.")


2026-06-23 15:45:15,717 - INFO - Splitting dataset chronologically...
2026-06-23 15:45:15,832 - INFO - Train samples (Occ): 55,504, Train samples (Reg, Rain > 0): 27,641
2026-06-23 15:45:15,975 - INFO - Scaler exported successfully.


# 5. METRICS & PLOTTING FUNCTIONS


In [6]:
def met_metrics(y_true, y_pred):
    hits = np.sum((y_pred == 1) & (y_true == 1))
    misses = np.sum((y_pred == 0) & (y_true == 1))
    false_alarms = np.sum((y_pred == 1) & (y_true == 0))
    correct_negatives = np.sum((y_pred == 0) & (y_true == 0))
    
    csi = hits / (hits + misses + false_alarms) if (hits + misses + false_alarms) > 0 else 0
    pod = hits / (hits + misses) if (hits + misses) > 0 else 0
    far = false_alarms / (hits + false_alarms) if (hits + false_alarms) > 0 else 0
    
    total = hits + misses + false_alarms + correct_negatives
    hits_random = ((hits + misses) * (hits + false_alarms)) / total if total > 0 else 0
    ets = (hits - hits_random) / (hits + misses + false_alarms - hits_random) if (hits + misses + false_alarms - hits_random) > 0 else 0
    hss = (2 * (hits * correct_negatives - misses * false_alarms)) / ((hits + misses) * (misses + correct_negatives) + (hits + false_alarms) * (false_alarms + correct_negatives)) if total > 0 else 0
    
    return {'CSI': float(csi), 'POD': float(pod), 'FAR': float(far), 'ETS': float(ets), 'HSS': float(hss)}

def evaluate_models(y_test_occ, prob_occ_uncal, prob_occ_cal, pred_occ_cal, y_test_reg, pred_reg):
    rmse = float(np.sqrt(mean_squared_error(y_test_reg, pred_reg)))
    mae = float(mean_absolute_error(y_test_reg, pred_reg))
    r2 = float(r2_score(y_test_reg, pred_reg))
    nse = float(1 - (np.sum((y_test_reg - pred_reg)**2) / (np.sum((y_test_reg - np.mean(y_test_reg))**2) + 1e-5)))
    r_pearson = float(np.corrcoef(y_test_reg, pred_reg)[0,1])
    kge = float(1 - np.sqrt((r_pearson - 1)**2 + (np.std(pred_reg)/(np.std(y_test_reg)+1e-5) - 1)**2 + (np.mean(pred_reg)/(np.mean(y_test_reg)+1e-5) - 1)**2))
    
    met = met_metrics(y_test_occ, pred_occ_cal)
    
    report = {
        'Regression_RainyDaysOnly': {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'NSE': nse, 'KGE': kge},
        'Classification': {
            'Accuracy': float(accuracy_score(y_test_occ, pred_occ_cal)),
            'Precision': float(precision_score(y_test_occ, pred_occ_cal, zero_division=0)),
            'Recall': float(recall_score(y_test_occ, pred_occ_cal, zero_division=0)),
            'F1': float(f1_score(y_test_occ, pred_occ_cal, zero_division=0)),
            'ROC_AUC': float(roc_auc_score(y_test_occ, prob_occ_cal)),
            'Brier_Uncalibrated': float(brier_score_loss(y_test_occ, prob_occ_uncal)),
            'Brier_Calibrated': float(brier_score_loss(y_test_occ, prob_occ_cal))
        },
        'Meteorological': met
    }
    return report

def plot_calibration(y_true, prob_uncal, prob_iso, prob_platt, save_path):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    f_op_uncal, m_pv_uncal = calibration_curve(y_true, prob_uncal, n_bins=10)
    f_op_iso, m_pv_iso = calibration_curve(y_true, prob_iso, n_bins=10)
    f_op_platt, m_pv_platt = calibration_curve(y_true, prob_platt, n_bins=10)
    
    plt.plot([0, 1], [0, 1], "k:", label="Perfectly calibrated")
    plt.plot(m_pv_uncal, f_op_uncal, "s-", alpha=0.5, label="Uncalibrated", color='orange')
    plt.plot(m_pv_iso, f_op_iso, "s-", label="Isotonic", color='blue')
    plt.plot(m_pv_platt, f_op_platt, "s-", label="Platt Scaling", color='green')
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Fraction of positives")
    plt.title("Reliability Diagram")
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    sns.histplot(prob_uncal, color='orange', alpha=0.2, label='Uncal', kde=True, bins=15)
    sns.histplot(prob_iso, color='blue', alpha=0.2, label='Isotonic', kde=True, bins=15)
    sns.histplot(prob_platt, color='green', alpha=0.2, label='Platt', kde=True, bins=15)
    plt.legend()
    plt.title("Probability Distribution")
    plt.tight_layout()
    plt.savefig(save_path, dpi=120)
    plt.close()

def plot_classification_diagnostics(y_true, pred, prob, save_path):
    plt.figure(figsize=(14, 4))
    
    plt.subplot(1, 3, 1)
    sns.heatmap(confusion_matrix(y_true, pred), annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Observed')
    
    plt.subplot(1, 3, 2)
    fpr, tpr, _ = roc_curve(y_true, prob)
    plt.plot(fpr, tpr, label=f'AUC={auc(fpr, tpr):.3f}', color='darkorange', linewidth=2)
    plt.plot([0,1],[0,1],'k--', color='gray')
    plt.legend(loc='lower right')
    plt.title('ROC Curve')
    plt.grid(True)
    
    plt.subplot(1, 3, 3)
    pr, rc, _ = precision_recall_curve(y_true, prob)
    plt.plot(rc, pr, label=f'AUC-PR={auc(rc, pr):.3f}', color='forestgreen', linewidth=2)
    plt.legend(loc='lower left')
    plt.title('Precision-Recall')
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=120)
    plt.close()

def plot_regression_diagnostics(y_true, pred, save_path):
    plt.figure(figsize=(14, 4))
    
    plt.subplot(1, 3, 1)
    plt.scatter(y_true, pred, alpha=0.4, color='royalblue', edgecolors='k', s=20)
    plt.plot([0, max(y_true)], [0, max(y_true)], 'r--', linewidth=2)
    plt.xlabel('Observed (mm)')
    plt.ylabel('Predicted (mm)')
    plt.title('Pred vs Obs (Rainy Only)')
    plt.grid(True)
    
    plt.subplot(1, 3, 2)
    residuals = y_true - pred
    sns.histplot(residuals, kde=True, color='crimson', bins=20)
    plt.title('Residual Distribution')
    plt.xlabel('Residual (mm)')
    plt.grid(True)
    
    plt.subplot(1, 3, 3)
    plt.plot(y_true[:80], label='Observed', color='black', alpha=0.7)
    plt.plot(pred[:80], label='Predicted', color='dodgerblue', alpha=0.9, linestyle='--')
    plt.legend()
    plt.title('Time Series Snippet')
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=120)
    plt.close()


# 6. BAYESIAN OPTUNA SEQUENTIAL TUNING


In [7]:
def create_seq(X, y, time_steps):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:(i + time_steps)])
        ys.append(y.iloc[i + time_steps])
    return np.array(Xs), np.array(ys)

def obj_lstm_occ(trial):
    keras.backend.clear_session()
    ts = trial.suggest_categorical('sequence_length', [8, 16, 24])
    hidden = trial.suggest_int('lstm_units', 32, 96, step=32)
    lr = trial.suggest_float('learning_rate', 5e-4, 5e-3, log=True)
    
    Xt, yt = create_seq(X_train_s, y_train['target_occurrence'], ts)
    Xv, yv = create_seq(X_val_s, y_val['target_occurrence'], ts)
    
    model = keras.Sequential([
        layers.Input(shape=(ts, Xt.shape[2])),
        layers.LSTM(hidden),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=keras.optimizers.Adam(lr), loss='binary_crossentropy')
    history = model.fit(Xt, yt, validation_data=(Xv, yv), epochs=EPOCHS_OCC, batch_size=BATCH_SIZE, verbose=0)
    return min(history.history['val_loss'])

# Run Optuna Study
logger.info(f"Running Optuna Bayesian Optimization for LSTM (Trials: {OPTUNA_TRIALS})...")
study_lstm_occ = optuna.create_study(direction='minimize')
study_lstm_occ.optimize(obj_lstm_occ, n_trials=OPTUNA_TRIALS)
p_lstm_occ = study_lstm_occ.best_params
p_lstm_reg = p_lstm_occ.copy()  # Share best hyperparameters with regression model

logger.info(f"Optuna Best Parameters: {p_lstm_occ}")


2026-06-23 15:45:16,061 - INFO - Running Optuna Bayesian Optimization for LSTM (Trials: 50)...


[I 2026-06-23 15:45:16,063] A new study created in memory with name: no-name-b46b67f1-63e7-4468-a04c-337aae0673de
2026-06-23 15:45:17.387657: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
[I 2026-06-23 15:50:33,896] Trial 0 finished with value: 0.47310206294059753 and parameters: {'sequence_length': 24, 'lstm_units': 32, 'learning_rate': 0.0011274519197190893}. Best is trial 0 with value: 0.47310206294059753.
[I 2026-06-23 15:54:21,388] Trial 1 finished with value: 0.473545104265213 and parameters: {'sequence_length': 16, 'lstm_units': 32, 'learning_rate': 0.004894677022282494}. Best is trial 0 with value: 0.47310206294059753.
[I 2026-06-23 16:06:33,194] Trial 2 finished with value: 0.47724902629852295 and parameters: {'sequence_length': 24, 'lstm_units': 96, 'learning_rate': 0.0017200144136154628}. Best is trial 0 with value: 0.47310206294059753.
[I 2026-06-23 16:18:29,457] Tri

2026-06-23 20:11:39,763 - INFO - Optuna Best Parameters: {'sequence_length': 24, 'lstm_units': 32, 'learning_rate': 0.0024825838363144452}


# 7. MODEL TRAINING & CALIBRATION (STAGE 1 & 2)


In [8]:
ts = p_lstm_occ['sequence_length']

# A. Rebuild sequences using optimal window length
Xt_o, yt_o = create_seq(X_train_s, y_train['target_occurrence'], ts)
Xv_o, yv_o = create_seq(X_val_s, y_val['target_occurrence'], ts)
Xte_o, yte_o = create_seq(X_test_s, y_test['target_occurrence'], ts)

Xt_r, yt_r = create_seq(X_train_reg_s, y_train_reg['target_amount'], ts)
Xv_r, yv_r = create_seq(X_val_reg_s, y_val_reg['target_amount'], ts)
Xte_r, yte_r = create_seq(X_test_reg_s, y_test_reg['target_amount'], ts)

# Keras Callbacks
cb_occ = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=0),
    keras.callbacks.ModelCheckpoint(str(OUTPUTS_DIR / 'model' / 'best_lstm_occ_satelit.keras'), monitor='val_loss', save_best_only=True)
]

cb_reg = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=0),
    keras.callbacks.ModelCheckpoint(str(OUTPUTS_DIR / 'model' / 'best_lstm_reg_satelit.keras'), monitor='val_loss', save_best_only=True)
]


## 1. Train Occurrence Model (Stage 1 Classifier)


In [9]:
# 1. Train Occurrence Model (Stage 1 Classifier)
logger.info("Training Stage 1 Classifier LSTM (Rain Occurrence)...")
keras.backend.clear_session()
clf = keras.Sequential([
    layers.Input(shape=(ts, Xt_o.shape[2])),
    layers.LSTM(p_lstm_occ['lstm_units']),
    layers.Dense(1, activation='sigmoid')
])
clf.compile(optimizer=keras.optimizers.Adam(p_lstm_occ['learning_rate']), loss='binary_crossentropy')
h_occ = clf.fit(Xt_o, yt_o, validation_data=(Xv_o, yv_o), epochs=EPOCHS_OCC, batch_size=BATCH_SIZE, verbose=1, callbacks=cb_occ)

# Get predictions
val_prob_uncal = clf.predict(Xv_o).flatten()
test_prob_uncal = clf.predict(Xte_o).flatten()


2026-06-23 20:11:42,561 - INFO - Training Stage 1 Classifier LSTM (Rain Occurrence)...
Epoch 1/50
434/434 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - loss: 0.5507 - val_loss: 0.5212 - learning_rate: 0.0025
Epoch 2/50
434/434 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - loss: 0.5239 - val_loss: 0.5125 - learning_rate: 0.0025
Epoch 3/50
434/434 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - loss: 0.5147 - val_loss: 0.5013 - learning_rate: 0.0025
Epoch 4/50
434/434 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - loss: 0.5081 - val_loss: 0.4981 - learning_rate: 0.0025
Epoch 5/50
434/434 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - loss: 0.5037 - val_loss: 0.4939 - learning_rate: 0.0025
Epoch 6/50
434/434 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - loss: 0.5002 - val_loss: 0.4906 - learning_rate: 0.0025
Epoch 7/50
434/434 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - loss: 0.4973 - val_loss: 0.4883 - learning_rate: 0.0025
Epoch 8/50
434/434 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - loss: 0.4951 - val_loss: 0.4864 - learning_rate: 0.0025
Epoch 9/50
434/434 ━━━━━━

## 2. Probability Calibration


In [10]:
# 2. Probability Calibration
logger.info("Performing probability calibration...")
iso_cal = IsotonicRegression(out_of_bounds='clip')
iso_cal.fit(val_prob_uncal, yv_o)
val_prob_iso = iso_cal.predict(val_prob_uncal)
brier_iso = brier_score_loss(yv_o, val_prob_iso)

platt_cal = LogisticRegression()
val_prob_2d = val_prob_uncal.reshape(-1, 1)
platt_cal.fit(val_prob_2d, yv_o)
val_prob_platt = platt_cal.predict_proba(val_prob_2d)[:, 1]
brier_platt = brier_score_loss(yv_o, val_prob_platt)

logger.info(f"Isotonic Brier Score: {brier_iso:.5f} | Platt Brier Score: {brier_platt:.5f}")

if brier_iso < brier_platt:
    logger.info("Selecting Isotonic Calibration.")
    selected_calibrator = "Isotonic"
    test_prob_cal = iso_cal.predict(test_prob_uncal)
    joblib.dump(iso_cal, OUTPUTS_DIR / 'model' / 'calibrator.pkl')
else:
    logger.info("Selecting Platt Calibration.")
    selected_calibrator = "Platt"
    test_prob_cal = platt_cal.predict_proba(test_prob_uncal.reshape(-1, 1))[:, 1]
    joblib.dump(platt_cal, OUTPUTS_DIR / 'model' / 'calibrator.pkl')

test_pred_cal = (test_prob_cal > 0.5).astype(int)
test_prob_iso = iso_cal.predict(test_prob_uncal)
test_prob_platt = platt_cal.predict_proba(test_prob_uncal.reshape(-1, 1))[:, 1]


2026-06-23 20:14:42,439 - INFO - Performing probability calibration...
2026-06-23 20:14:42,466 - INFO - Isotonic Brier Score: 0.15248 | Platt Brier Score: 0.15659
2026-06-23 20:14:42,469 - INFO - Selecting Isotonic Calibration.


## 3. Train Rain Amount Model (Stage 2 Regressor)


In [11]:
# 3. Train Rain Amount Model (Stage 2 Regressor)
logger.info("Training Stage 2 Regressor LSTM (Rain Amount)...")
keras.backend.clear_session()
reg = keras.Sequential([
    layers.Input(shape=(ts, Xt_r.shape[2])),
    layers.LSTM(p_lstm_reg['lstm_units']),
    layers.Dense(1, activation='linear')
])
reg.compile(optimizer=keras.optimizers.Adam(p_lstm_reg['learning_rate']), loss=keras.losses.Huber())
h_reg = reg.fit(Xt_r, yt_r, validation_data=(Xv_r, yv_r), epochs=EPOCHS_REG, batch_size=BATCH_SIZE, verbose=1, callbacks=cb_reg)

# Predict regression
test_pred_reg = np.maximum(0, reg.predict(Xte_r).flatten())


2026-06-23 20:14:43,109 - INFO - Training Stage 2 Regressor LSTM (Rain Amount)...
Epoch 1/200
216/216 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - loss: 0.9355 - val_loss: 0.8705 - learning_rate: 0.0025
Epoch 2/200
216/216 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 0.8945 - val_loss: 0.8606 - learning_rate: 0.0025
Epoch 3/200
216/216 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 0.8821 - val_loss: 0.8522 - learning_rate: 0.0025
Epoch 4/200
216/216 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 0.8750 - val_loss: 0.8482 - learning_rate: 0.0025
Epoch 5/200
216/216 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 0.8696 - val_loss: 0.8440 - learning_rate: 0.0025
Epoch 6/200
216/216 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 0.8653 - val_loss: 0.8410 - learning_rate: 0.0025
Epoch 7/200
216/216 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 0.8617 - val_loss: 0.8386 - learning_rate: 0.0025
Epoch 8/200
216/216 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - loss: 0.8582 - val_loss: 0.8361 - learning_rate: 0.0025
Epoch 9/200
216/216 ━━

# 8. PIPELINE EVALUATION AND METRIC EXPORT


In [12]:
logger.info("Evaluating models and generating plots...")
report = evaluate_models(yte_o, test_prob_uncal, test_prob_cal, test_pred_cal, yte_r, test_pred_reg)

report['Selected_Calibrator'] = selected_calibrator
report['Optuna_Parameters'] = p_lstm_occ

# Save metrics report
with open(OUTPUTS_DIR / 'metrics' / 'report.json', 'w') as f:
    json.dump(report, f, indent=4)
logger.info(f"Metrics Report exported to: {OUTPUTS_DIR / 'metrics' / 'report.json'}")

# Generate diagrams and plots
plot_calibration(yte_o, test_prob_uncal, test_prob_iso, test_prob_platt, OUTPUTS_DIR / 'plots' / 'calibration_curves.png')
plot_classification_diagnostics(yte_o, test_pred_cal, test_prob_cal, OUTPUTS_DIR / 'plots' / 'classification_diagnostics.png')
plot_regression_diagnostics(yte_r, test_pred_reg, OUTPUTS_DIR / 'plots' / 'regression_residuals.png')

# Save training history loss curves
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(h_occ.history['loss'], label='Train')
plt.plot(h_occ.history['val_loss'], label='Val')
plt.title('Stage 1 (Classifier) Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(h_reg.history['loss'], label='Train')
plt.plot(h_reg.history['val_loss'], label='Val')
plt.title('Stage 2 (Regressor) Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'plots' / 'training_loss_curves.png', dpi=120)
plt.close()

logger.info("\n=== SATELLITE-ENHANCED RAINFOREST PIPELINE COMPLETED SUCCESSFULLY! ===")
print(json.dumps(report, indent=4))


2026-06-23 20:15:52,427 - INFO - Evaluating models and generating plots...
2026-06-23 20:15:52,450 - INFO - Metrics Report exported to: /kaggle/working/outputs/metrics/report.json


/tmp/ipykernel_16/875810683.py:82: UserWarning: color is redundantly defined by the 'color' keyword argument and the fmt string "k--" (-> color='k'). The keyword argument will take precedence.
  plt.plot([0,1],[0,1],'k--', color='gray')


2026-06-23 20:15:54,597 - INFO - 
=== SATELLITE-ENHANCED RAINFOREST PIPELINE COMPLETED SUCCESSFULLY! ===
{
    "Regression_RainyDaysOnly": {
        "RMSE": 4.294465262224268,
        "MAE": 1.5264446866932628,
        "R2": 0.08792741631790535,
        "NSE": 0.08792741655707115,
        "KGE": -0.0886829352937546
    },
    "Classification": {
        "Accuracy": 0.7617403314917127,
        "Precision": 0.7784956160590678,
        "Recall": 0.8892988929889298,
        "F1": 0.8302165354330708,
        "ROC_AUC": 0.8252050152419156,
        "Brier_Uncalibrated": 0.15864714540472305,
        "Brier_Calibrated": 0.1582100425790717
    },
    "Meteorological": {
        "CSI": 0.7097181320992848,
        "POD": 0.8892988929889298,
        "FAR": 0.22150438394093216,
        "ETS": 0.2793923388915875,
        "HSS": 0.436757873872594
    },
    "Selected_Calibrator": "Isotonic",
    "Optuna_Parameters": {
        "sequence_length": 24,
        "lstm_units": 32,
        "learning_rate": 0.